# Notebook 4: Exploratory Data Analysis

This notebook explores the training split only. It checks data types, missing values, numerical distributions, categorical value counts, the relationship between features and the label, and date/geography effects.

**Reads:** `train.csv`  
**Artifacts produced:** `eda_hist_*.png`, `eda_late_ratio_by_state.png`, `eda_findings.txt`

In [1]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
train = pd.read_csv("train.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
])

In [3]:
findings = []

In [4]:
print("Shape:", train.shape)
findings.append(f"Training set shape: {train.shape}")

Shape: (67526, 21)


In [5]:
print("\nData types:")
print(train.dtypes)


Data types:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
customer_unique_id                       object
customer_zip_code_prefix                  int64
customer_city                            object
customer_state                           object
n_items                                 float64
total_price                             float64
total_freight                           float64
product_category_name                    object
product_category_name_english            object
total_payment_value                     float64
n_payment_installments                  float64
payment_type                             object
is_late                    

In [6]:
print("\nMemory usage (MB):", train.memory_usage(deep=True).sum() / 1e6)


Memory usage (MB): 46.655693


In [7]:
# Missing values
missing = train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nMissing values per column:")
print(missing)
findings.append(f"Columns with missing values: {list(missing.index)}")


Missing values per column:
product_category_name_english    964
product_category_name            951
order_approved_at                 11
dtype: int64


In [8]:
# Numerical columns
numeric_cols = ["total_price", "total_freight", "n_items", "total_payment_value", "n_payment_installments"]
print("\nNumerical summary:")
print(train[numeric_cols].describe())


Numerical summary:
        total_price  total_freight       n_items  total_payment_value  \
count  67526.000000   67526.000000  67526.000000         67526.000000   
mean     137.535552      22.814645      1.141723           160.378374   
std      211.198679      22.064300      0.534479           221.246034   
min        0.850000       0.000000      1.000000             9.590000   
25%       45.970000      13.830000      1.000000            62.110000   
50%       86.900000      17.150000      1.000000           105.370000   
75%      149.900000      23.990000      1.000000           176.630000   
max    13440.000000    1794.960000     20.000000         13664.080000   

       n_payment_installments  
count            67526.000000  
mean                 2.918046  
std                  2.700601  
min                  0.000000  
25%                  1.000000  
50%                  2.000000  
75%                  4.000000  
max                 24.000000  


In [9]:
for col in numeric_cols:
    plt.figure()
    train[col].dropna().plot(kind="hist", bins=50, title=col)
    plt.xlabel(col)
    plt.savefig(f"eda_hist_{col}.png")
    plt.close()

In [10]:
skewed = train[numeric_cols].skew()
print("\nSkewness:")
print(skewed)
findings.append(f"Most skewed numeric column: {skewed.abs().idxmax()}")


Skewness:
total_price               10.443177
total_freight             13.520443
n_items                    7.241030
total_payment_value        9.879775
n_payment_installments     1.589026
dtype: float64


In [11]:
# Categorical columns
categorical_cols = ["order_status", "customer_state", "payment_type", "product_category_name_english"]
for col in categorical_cols:
    if col in train.columns:
        print(f"\nValue counts for {col}:")
        print(train[col].value_counts().head(10))


Value counts for order_status:
order_status
delivered    67526
Name: count, dtype: int64

Value counts for customer_state:
customer_state
SP    28469
RJ     8554
MG     7975
RS     3718
PR     3454
SC     2514
BA     2272
DF     1430
ES     1399
GO     1335
Name: count, dtype: int64

Value counts for payment_type:
payment_type
credit_card    50905
boleto         13483
voucher         2076
debit_card      1062
Name: count, dtype: int64

Value counts for product_category_name_english:
product_category_name_english
bed_bath_table           6450
health_beauty            5920
sports_leisure           5310
computers_accessories    4594
furniture_decor          4266
housewares               3962
watches_gifts            3820
telephony                2871
auto                     2673
toys                     2654
Name: count, dtype: int64


In [12]:
# Relation with label
print("\nLate ratio by payment type:")
by_payment = train.groupby("payment_type")["is_late"].mean().sort_values(ascending=False)
print(by_payment)


Late ratio by payment type:
payment_type
boleto         0.090113
debit_card     0.083804
credit_card    0.079049
voucher        0.072254
Name: is_late, dtype: float64


In [13]:
print("\nLate ratio by customer state (top 10 by count):")
top_states = train["customer_state"].value_counts().head(10).index
by_state = train[train["customer_state"].isin(top_states)].groupby("customer_state")["is_late"].mean().sort_values(ascending=False)
print(by_state)
findings.append(f"State with highest late ratio (top 10 by volume): {by_state.idxmax()}")


Late ratio by customer state (top 10 by count):
customer_state
BA    0.136444
RJ    0.133972
ES    0.128663
SC    0.101034
GO    0.082397
RS    0.071544
DF    0.067832
SP    0.058871
MG    0.055298
PR    0.050087
Name: is_late, dtype: float64


In [14]:
plt.figure()
by_state.plot(kind="bar", title="Late ratio by customer state")
plt.ylabel("late ratio")
plt.tight_layout()
plt.savefig("eda_late_ratio_by_state.png")
plt.close()

In [15]:
# Dates: delivery time in days, weekday effect
train["delivery_days"] = (train["order_delivered_customer_date"] - train["order_purchase_timestamp"]).dt.days
train["purchase_weekday"] = train["order_purchase_timestamp"].dt.dayofweek

In [16]:
print("\nDelivery days summary:")
print(train["delivery_days"].describe())


Delivery days summary:
count    67526.000000
mean        12.100080
std          9.612969
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64


In [17]:
plt.figure()
train["delivery_days"].dropna().plot(kind="hist", bins=50, title="Delivery days")
plt.savefig("eda_hist_delivery_days.png")
plt.close()

In [18]:
print("\nLate ratio by purchase weekday (0=Mon):")
by_weekday = train.groupby("purchase_weekday")["is_late"].mean()
print(by_weekday)


Late ratio by purchase weekday (0=Mon):
purchase_weekday
0    0.089867
1    0.083303
2    0.077241
3    0.077419
4    0.086776
5    0.073768
6    0.075984
Name: is_late, dtype: float64


In [19]:
# Geography: state distance proxy using zip code prefix difference is skipped for time,
# state-level analysis above already covers the geography angle.

In [20]:
findings.append("Delivery time and customer state show visible differences in late ratio, "
                 "suggesting both are useful features for the model.")

In [21]:
with open("eda_findings.txt", "w") as f:
    f.write("\n".join(findings))

In [22]:
print("\nArtifacts saved: eda_hist_*.png, eda_late_ratio_by_state.png, eda_findings.txt")


Artifacts saved: eda_hist_*.png, eda_late_ratio_by_state.png, eda_findings.txt
